In [2]:
import pandas as pd
import numpy as np
import joblib

# 1. โหลด Model และ Scaler (ควรทำนอกฟังก์ชัน เพื่อโหลดทีเดียวตอนเริ่ม Server)
# ตรวจสอบ path ไฟล์ให้ถูกนะครับ
try:
    model = joblib.load(r'C:\Users\nongt\OneDrive\Desktop\model\best_model_Random_Forest_Normalized.pkl')
    scaler = joblib.load(r'C:\Users\nongt\OneDrive\Desktop\model\landslide_scaler.pkl')
    print("✅ Load Model Success")
except:
    print("❌ Error: หาไฟล์ Model ไม่เจอ")

def predict_landslide(daily_rain_list, static_features):
    """
    daily_rain_list: List ปริมาณฝนรายวันย้อนหลัง 10 วัน [Day1_LATEST, Day2, ..., Day10]
                     (รับค่าตรงๆ มาจากฟังก์ชัน Open-Meteo ได้เลย ไม่ต้องเป็นรายชั่วโมง)
    static_features: Dictionary ข้อมูลพื้นที่
    """

    # --- PART 1: เตรียมข้อมูลฝน (แก้ใหม่ให้รับรายวันโดยตรง) ---
    # ไม่ต้อง Loop บวก 24 ชม. แล้ว เพราะ Open-Meteo คิดมาให้แล้ว
    if len(daily_rain_list) < 10:
        return {"error": "ข้อมูลฝนไม่ครบ 10 วัน"}
    
    daily_rains = daily_rain_list # ใช้ได้เลย

    # --- PART 2: คำนวณ Feature (Feature Engineering) ---
    features = {}

    # 2.1 ใส่ค่าฝนรายวัน (Day 1 - 10)
    for i in range(10):
        features[f'CHIRPS_Day_{i+1}'] = daily_rains[i]

    # 2.2 คำนวณฝนสะสม (Prior)
    features['Rain_3D_Prior'] = sum(daily_rains[0:3]) # 3 วันล่าสุด
    features['Rain_5D_Prior'] = sum(daily_rains[0:5])
    features['Rain_7D_Prior'] = sum(daily_rains[0:7])

    # 2.3 ดึงค่าพื้นที่และคำนวณ Interaction
    try:
        slope = static_features['Slope']
        features['Slope_Extracted'] = slope
        features['Rain3D_x_Slope'] = features['Rain_3D_Prior'] * slope
        features['Rain5D_x_Slope'] = features['Rain_5D_Prior'] * slope
        features['Rain7D_x_Slope'] = features['Rain_7D_Prior'] * slope

        # 2.4 จัดการ Road Zone
        dist = static_features['Distance_to_Road']
        if dist <= 50:
            road_zone = 1
        elif dist <= 100:
            road_zone = 2
        elif dist <= 200:
            road_zone = 3
        elif dist <= 500:
            road_zone = 4
        else:
            road_zone = 5
        features['Road_Zone'] = road_zone

        # 2.5 ใส่ค่า Static อื่นๆ
        features['Elevation_Extracted'] = static_features['Elevation']
        features['Aspect_Extracted'] = static_features['Aspect']
        features['MODIS_LC'] = static_features['MODIS_LC']
        features['NDVI'] = static_features['NDVI']
        features['NDWI'] = static_features['NDWI']
        features['TWI'] = static_features['TWI']
        features['Soil_Type'] = static_features['Soil_Type']
        
    except KeyError as e:
        return {"error": f"ข้อมูล Static Features ไม่ครบ: ขาด key {e}"}

    # --- PART 3: จัดเรียง Column และ Normalize ---
    
    # ลำดับต้องเป๊ะตามตอน Train 100%
    columns_order = [
        'CHIRPS_Day_1', 'CHIRPS_Day_2', 'CHIRPS_Day_3', 'CHIRPS_Day_4', 'CHIRPS_Day_5',
        'CHIRPS_Day_6', 'CHIRPS_Day_7', 'CHIRPS_Day_8', 'CHIRPS_Day_9', 'CHIRPS_Day_10',
        'Elevation_Extracted', 'Slope_Extracted', 'Aspect_Extracted',
        'MODIS_LC', 'NDVI', 'NDWI', 'TWI', 'Soil_Type',
        'Road_Zone',
        'Rain_3D_Prior', 'Rain_5D_Prior', 'Rain_7D_Prior',
        'Rain3D_x_Slope', 'Rain5D_x_Slope', 'Rain7D_x_Slope'
    ]

    # สร้าง DataFrame
    try:
        df_input = pd.DataFrame([features], columns=columns_order)
    except Exception as e:
        # กรณีลืม column ไหนไป มันจะฟ้องตรงนี้
        return {"error": f"สร้าง DataFrame ไม่ผ่าน: {e}"}

    # Scale ข้อมูล
    X_scaled = scaler.transform(df_input)

    # --- PART 4: ทำนายผล (Prediction) ---
    
    # ดึงความน่าจะเป็น (Probability)
    risk_probability = model.predict_proba(X_scaled)[0][1] 

    # --- PART 5: แปลงเป็นระดับความเสี่ยง (Thresholding) ---
    
    result = {}
    result['probability_score'] = round(risk_probability * 100, 2)

    if risk_probability < 0.40:
        result['level'] = "Low"
        result['color'] = "Green"
        result['message'] = "ปกติ: เฝ้าระวังตามรอบปกติ"
        
    elif 0.40 <= risk_probability < 0.70:
        result['level'] = "Medium"
        result['color'] = "Yellow"
        result['message'] = "เฝ้าระวัง: ฝนสะสมเริ่มสูง แจ้งเตือนมิสเตอร์เตือนภัย"
        
    else:
        result['level'] = "High"
        result['color'] = "Red"
        result['message'] = "อันตราย: เสี่ยงดินถล่มสูง เตรียมอพยพ!"

    return result

✅ Load Model Success


c:\Users\nongt\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.4.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\nongt\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.4.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\nongt\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.4.2 when using version 1.6.1. This might 

In [3]:
# จำลองข้อมูลจาก Open-Meteo (รายวัน 10 วันล่าสุด)
mock_rain_from_api = [50.5, 40.0, 10.0, 0.0, 0.0, 5.0, 0.0, 0.0, 0.0, 0.0]

# จำลองข้อมูลพื้นที่
mock_static_data = {
    'Slope': 45,
    'Elevation': 600,
    'Distance_to_Road': 20, # จะถูกแปลงเป็น Zone 1
    'Aspect': 150,
    'MODIS_LC': 10,
    'NDVI': 0.6,
    'NDWI': 0.2,
    'TWI': 12.5,
    'Soil_Type': 2
}

# เรียกใช้งาน
prediction = predict_landslide(mock_rain_from_api, mock_static_data)
print(prediction)

{'probability_score': np.float64(69.0), 'level': 'Medium', 'color': 'Yellow', 'message': 'เฝ้าระวัง: ฝนสะสมเริ่มสูง แจ้งเตือนมิสเตอร์เตือนภัย'}
